# STAGE 10A · GO / NO-GO gate for acquisition-conditional specialisation

### ~20 min · ~0.6 CU · `best.pt` never written

---

## Why this exists

Stage 9B proved **invariance fails**: driving projection AUC to **0.5000** (complete invariance, beyond the published method's 0.61) closed only **13.3%** of the gap and cost **0.0789 AUROC**. The disparity is not a learned shortcut — AP films are *intrinsically harder to read*.

If the two projections are genuinely different problems, the fix is not to hide the difference but to let the model **specialise**. One shared head must compromise between two distributions whose statistics differ sharply (cardiomegaly prevalence **62% AP vs 32% PA**), and that compromise costs accuracy on **both** groups.

## But specialisation might not help — so we test it cheaply first

Freeze the Stage 5 trunk, extract features once, and fit three linear probes:

| arm | design |
|---|---|
| **A shared** | 1024-d features → one head *(the baseline)* |
| **B shared+acq** | features ++ the 8-d acquisition vector *(cheapest conditioning)* |
| **C per-projection** | separate AP and PA heads *(full specialisation)* |

**If C cannot beat A, the idea is dead and you have spent 0.6 CU instead of 6.**

> Arm C is handicapped on purpose: it splits the training data between two heads, so each sees roughly half. Winning *despite* that is strong evidence. Losing narrowly is ambiguous and will be reported as such, not spun.

---
# 0 · Config & safety

In [ ]:
import os, sys, json, time, hashlib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch

N_PROBE_TRAIN = 20000   # subsample for probe fitting -- plenty for a linear head
BATCH         = 64
NUM_WORKERS   = 2

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
else: print('  Drive already mounted')
PROJECT  = Path('/content/drive/MyDrive/Component_01')
IMG_ROOT = Path('/content/cardio_image_384')
TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
MANIFEST = PROJECT / 'training_manifest'
METADATA = PROJECT / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
S5_CKPT  = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
S6CACHE  = PROJECT / 'reports' / 'stage6' / 'cache'
OUT      = PROJECT / 'reports' / 'stage10'; OUT.mkdir(parents=True, exist_ok=True)
FEAT     = OUT / 'features'; FEAT.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT))

# ---- HARD SAFETY GUARDS ----
S5DIR = (PROJECT / 'checkpoints' / 'stage5').resolve()
for p in (OUT, FEAT):
    r = p.resolve()
    assert r != S5DIR and S5DIR not in r.parents, 'output would collide with stage5!'
SHA_BEFORE = hashlib.sha256(S5_CKPT.read_bytes()).hexdigest()
print('  best.pt SHA-256:', SHA_BEFORE[:40])
print('  outputs ->', OUT)
DEV = 'cuda'; assert torch.cuda.is_available(), 'select an L4 GPU'
import subprocess
print(' ', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

---
# 0b · Stage the images

Skipped automatically if already present, so **Restart session** is free.

In [ ]:
if not IMG_ROOT.exists():
    import shutil
    t0 = time.time(); lt = Path('/content/cardio_384.tar')
    if not lt.exists():
        assert TAR.exists(), 'tar not found: ' + str(TAR)
        print('  copying %.1f GB off Drive...' % (TAR.stat().st_size / 1e9))
        shutil.copy(TAR, lt)
    subprocess.run(['tar', '-xf', str(lt), '-C', '/content'], check=True)
    print('  staged in %.1f min' % ((time.time() - t0) / 60))
    try: lt.unlink()
    except OSError: pass
if not IMG_ROOT.exists():
    cand = [p for p in Path('/content').glob('*') if p.is_dir() and (p/'test').is_dir()]
    assert cand, 'extraction produced no directory containing test/'
    IMG_ROOT = cand[0]
print('  images:', IMG_ROOT, IMG_ROOT.exists())

---
# 1 · Gate: self-tests

**104 tests across four modules.**

In [ ]:
import stage6_acr as acr, stage9_fairness as s9
import stage9b_gradrev as s9b, stage10_conditional as s10
tot = 0
for nm, mod in (('stage6_acr', acr), ('stage9_fairness', s9),
                ('stage9b_gradrev', s9b), ('stage10_conditional', s10)):
    p, f = mod._selftest(verbose=False)
    print('  %-20s %3d passed  %d failed' % (nm, p, f))
    assert f == 0, nm + ' FAILED'
    tot += p
print('\n  ALL %d TESTS PASSED' % tot)

---
# 2 · Extract frozen features

The **only** GPU work. The Stage 5 trunk is frozen — this is exactly the representation `best.pt` uses, so any probe difference is attributable to the head alone.

~45,558 images, ~5 min on L4. Cached to Drive and **resumable**.

In [ ]:
from cxr_transforms import build_transform
from torch.utils.data import DataLoader
PATH = s10.PATHOLOGIES
man  = {s: pd.read_csv(MANIFEST / ('manifest_' + s + '.csv'), low_memory=False)
        for s in ('train', 'val', 'test')}
meta = pd.read_csv(METADATA, low_memory=False)

ck = torch.load(S5_CKPT, map_location='cpu', weights_only=False)   # READ ONLY
assert list(ck.get('pathologies', PATH)) == list(PATH), 'pathology ORDER differs!'
model = s10.CXRConditional(len(PATH), 'shared')
print('  ', model.load_stage5(ck, use_ema=True))
model = model.eval().to(DEV).to(memory_format=torch.channels_last)
del ck
TF = build_transform('test')

@torch.no_grad()
def extract(split):
    f = FEAT / ('feat_' + split + '.npy')
    if f.exists():
        X = np.load(f)
        if len(X) == len(man[split]):
            print('  %-5s cached (%d x %d)' % (split, *X.shape)); return X
    ds = s9b.CXRDataset(man[split], IMG_ROOT, TF, PATH)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    out, t0 = [], time.time()
    for i, (x, _, _, _) in enumerate(dl):
        x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            out.append(model.backbone(x).float().cpu().numpy())
        n = sum(len(o) for o in out); el = time.time() - t0
        print('\r  %-5s %d/%d  %.0f img/s' % (split, n, len(ds), n/max(el,1e-9)), end='')
    X = np.concatenate(out).astype(np.float32)
    np.save(f, X)
    print('\r  %-5s %d x %d in %.1f min' % (split, *X.shape, (time.time()-t0)/60))
    return X

T0 = time.time()
X = {s: extract(s) for s in ('train', 'val', 'test')}
print('  total %.1f min' % ((time.time() - T0) / 60))

Y = {s: man[s][PATH].astype(int).reset_index(drop=True) for s in X}
A = {}
for s in X:
    a = acr.metadata_acquisition(man[s], meta)
    ff = S6CACHE / ('imgfeat_' + s + '.parquet')
    if ff.exists() and len(pd.read_parquet(ff)) == len(man[s]):
        a = pd.concat([a.reset_index(drop=True),
                       pd.read_parquet(ff).reset_index(drop=True)], axis=1)
    else:   # train has no Stage 6 cache -> pixel proxies default to 0
        for c in s10.ACQ_FEATURES:
            if c not in a.columns: a[c] = 0.0
    A[s] = a
    assert len(X[s]) == len(Y[s]) == len(A[s]), 'row mismatch in ' + s
    print('  %-5s X %s  AP %d / PA %d' % (s, X[s].shape,
          (A[s].is_AP > .5).sum(), (A[s].is_AP <= .5).sum()))

---
# 3 · THE GATE ★

Three linear probes on identical frozen features. Fit on train, scored on **test**.

**Arm C beating arm A is the GO signal.**

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(X['train']), min(N_PROBE_TRAIN, len(X['train'])), replace=False)
print('  fitting probes on %d train samples (~5-10 min) ...' % len(idx))
t0 = time.time()
R = s10.compare_probes(X['train'][idx], Y['train'].iloc[idx].reset_index(drop=True),
                       A['train'].iloc[idx].reset_index(drop=True),
                       X['test'], Y['test'], A['test'], PATH)
print('  done in %.1f min\n' % ((time.time() - t0) / 60))

print('=' * 84)
print('  LINEAR PROBE ABLATION  (frozen Stage 5 features, test n=%d)' % len(X['test']))
print('=' * 84)
print('  %-24s %12s %12s' % ('arm', 'mean AUROC', 'vs shared'))
print('  ' + '-' * 81)
base = R['A_shared']['mean_auroc']
NAMES = {'A_shared': 'A  shared head (baseline)',
         'B_shared_plus_acq': 'B  shared + acquisition',
         'C_per_projection': 'C  per-projection heads'}
for t in ('A_shared', 'B_shared_plus_acq', 'C_per_projection'):
    m = R[t]['mean_auroc']
    print('  %-24s %12.4f %+12.4f' % (NAMES[t], m, m - base))
print('  ' + '-' * 81)
print()
print('  %-20s %10s %10s %10s' % ('pathology', 'A shared', 'B +acq', 'C per-proj'))
print('  ' + '-' * 55)
for k in PATH:
    print('  %-20s %10.4f %10.4f %10.4f'
          % (k, R['A_shared']['per_pathology'][k],
             R['B_shared_plus_acq']['per_pathology'][k],
             R['C_per_projection']['per_pathology'][k]))
print()
v = R['verdict']
print('  gain from per-projection heads : %+.4f' % v['gain_conditional'])
print('  gain from acquisition features : %+.4f' % v['gain_acq_features'])
print('  best arm                       : %s' % v['best'])
print()
if v['go']:
    print('  >>> GO. Specialisation helps even on FROZEN features, and arm C is')
    print('      handicapped (each head sees ~half the data). Fine-tuning the trunk')
    print('      end-to-end should do better. Proceed to Stage 10B.')
else:
    print('  >>> NO-GO. Conditioning does not beat a shared head by a margin worth')
    print('      chasing. Do NOT spend the ~5 CU on Stage 10B. Report this as a')
    print('      negative result: it completes the story -- neither invariance (9B)')
    print('      nor specialisation improves on the shared baseline, so the AP/PA')
    print('      gap is irreducible at the representation level.')

---
# 4 · Save & verify

In [ ]:
from datetime import datetime
res = dict(stage='10a', timestamp=datetime.now().isoformat(),
           n_probe_train=int(len(idx)), n_test=int(len(X['test'])),
           probes={t: R[t] for t in ('A_shared','B_shared_plus_acq','C_per_projection')},
           verdict=R['verdict'])
(OUT / 'stage10a_probe.json').write_text(
    json.dumps(res, indent=2, default=float), encoding='utf-8')
print('  saved', OUT / 'stage10a_probe.json')
print('  features cached in', FEAT, '(reused by Stage 10B)')
SHA_AFTER = hashlib.sha256(S5_CKPT.read_bytes()).hexdigest()
assert SHA_AFTER == SHA_BEFORE, 'best.pt WAS MODIFIED -- must never happen'
print('  *** best.pt VERIFIED BYTE-IDENTICAL ***')

---
# What this decides

| outcome | next step | cost |
|---|---|---|
| **GO** — arm C beats arm A | build Stage 10B, fine-tune end-to-end | ~5 CU |
| **NO-GO** | stop; report as the third arm of the ablation | **0 CU** |

A NO-GO is not a wasted result. Combined with 9A and 9B it gives a complete three-way finding: *the AP/PA gap resists thresholding (provably), invariance (measured), and specialisation (measured)* — evidence that the disparity is irreducible at the representation level and reflects genuine information loss in AP acquisition.